RandamForest:決定木４本で\
DecisionTree：決定木１本で

In [ ]:
import pandas as pd
import numpy as np

# df = pd.read_csv('data/relax_class.csv')
# df= pd.read_csv('C:/Users/hiros/OneDrive - 学校法人立命館/デスクトップ/TWILITE data/data1/acc_class.csv')
df = pd.read_csv('C:/Users/is0671vx/OneDrive - 学校法人立命館/デスクトップ/TWILITE data/data1/acc_class.csv')
df

In [ ]:
df.describe()

In [ ]:
from numpy.polynomial.polynomial import polyfit
from statsmodels.tsa.ar_model import AutoReg
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
# from sklearn.model_selection import train_test_split
from pykalman import KalmanFilter


def kalman_filter_series(series):
    # カルマンフィルタの初期化（状態は観測と同次元で、初期値は最初の観測値）
    kf = KalmanFilter(initial_state_mean=series.iloc[0], n_dim_obs=1)

    # フィルタ処理を実行：ノイズを考慮しながら系列の滑らかな推定値を返す
    state_means, _ = kf.filter(series.values.reshape(-1, 1))

    # フィルタ結果を pd.Series に変換
    return pd.Series(state_means.flatten(), index=series.index)


# 特徴抽出関数：多項式近似 or ARモデル
def vectorize_sequence(df, method='poly', degree_or_lags=3):
    features = {}
    t = df['sequence_counter'].values
    
    # もし、カラムにclassがあれば削除,なければそのまま
    if 'class' in df.columns:
        # cols = df.columns.drop(['sequence_id','sequence_counter', 'time','class'])
        cols = df.columns.drop(['sequence_counter', 'time','class'])
    else:
        # cols = df.columns.drop(['sequence_id','sequence_counter', 'time'])
        cols = df.columns.drop(['sequence_counter', 'time'])

    cols = ['acc_x','acc_y','acc_z','ax_f','ay_f','az_f','vx','vy','vz']

    for col in cols:
        y = df[col].values

        if method == 'poly':
            try:
                coefs = polyfit(t, y, degree_or_lags)
            except:
                coefs = np.zeros(degree_or_lags + 1)
            for i, c in enumerate(coefs):
                features[f'{col}_coef{i}'] = c

        elif method == 'ar':
            try:
                model = AutoReg(y, lags=degree_or_lags, old_names=False).fit()
                for i, coef in enumerate(model.params):
                    features[f'{col}_ar{i}'] = coef
            except:
                for i in range(degree_or_lags + 1):
                    features[f'{col}_ar{i}'] = 0.0
                    
        elif method == 'kalman':
            try:
                y_kf = kalman_filter_series(df[col])
                features[f'{col}_kf_mean'] = y_kf.mean()
                features[f'{col}_kf_std'] = y_kf.std()
                features[f'{col}_kf_min'] = y_kf.min()
                features[f'{col}_kf_max'] = y_kf.max()

                try:
                    ac = y_kf.autocorr(lag=1)
                    features[f'{col}_kf_autocorr'] = ac if not np.isnan(ac) else 0.0
                except:
                    features[f'{col}_kf_autocorr'] = 0.0
            except:
                features[f'{col}_mean'] = 0.0
                features[f'{col}_std'] = 0.0
                features[f'{col}_min'] = 0.0
                features[f'{col}_max'] = 0.0                    
                    
    return features

# クラスごとにセグメント分割し、ベクトル化とモデル学習を行う関数
def train_rf_by_class_segments(df_all, method='poly', degree_or_lags=3, n_estimators=4,
                                segment_length=50, step=50):
    """
    クラス単位でセグメント分割 → ベクトル化 → ランダムフォレスト学習

    Parameters:
    - df_all: 全データ（class列が必要）
    - method: 'poly' or 'ar'
    - degree_or_lags: 多項式次数 or ARラグ
    - n_estimators: 決定木の本数
    - segment_length: セグメントの長さ
    - step: セグメントのステップ幅（オーバーラップ処理）
    """
    feature_list = []
    label_list = []

    df_all = df_all[df_all['class'].notnull()].copy()
    df_all['class'] = df_all['class'].astype(int)

    for label in df_all['class'].unique():
        df_label = df_all[df_all['class'] == label]
        for start in range(0, len(df_label) - segment_length + 1, step):
            segment = df_label.iloc[start:start + segment_length]
            vec = vectorize_sequence(segment, method=method, degree_or_lags=degree_or_lags)
            vec['label'] = label
            feature_list.append(vec)

    df_vec = pd.DataFrame(feature_list)
    X = df_vec.drop(columns='label')
    y = (df_vec['label'] == 0).astype(int)  # 指定したクラスを1、それ以外を0
    # y = df_vec['label'].astype(int)   #多クラス分類用


    model = RandomForestClassifier(n_estimators=n_estimators, random_state=42)
    # model = DecisionTreeClassifier(random_state=42,criterion='entropy')
    model.fit(X, y)

    # return model, X_train, X_test, y_train, y_test, df_vec.columns.drop('label')
    return model, X ,y, df_vec.columns.drop('label')


In [ ]:
# パラメータ指定
method = 'kalman'  # 'poly' or 'ar' or 'kalman'
degree_or_lags = 3  # 多項式次数 or ARラグ数
n_estimators = 4  # 決定木の本数
segment_length = 10  # セグメントの長さ
step = 1  # セグメントのステップ幅

In [ ]:
# モデルの学習結果を確認
model, X, y, feature_names = train_rf_by_class_segments(df, method=method, degree_or_lags=degree_or_lags, n_estimators=4,
                                                        segment_length=segment_length, step=step)
print("学習済みモデル:", model)
print("特徴量名:", feature_names)
print("特徴量のサンプル:\n", X.head())
print("ラベルのサンプル:\n", y.head())

In [ ]:
# from matplotlib import pyplot as plt
# import seaborn as sns


# # 特徴量の重要度を可視化
# importances = model.feature_importances_
# indices = np.argsort(importances)[::-1]
# plt.figure(figsize=(12, 6))
# plt.title("Feature Importances")
# plt.bar(range(len(importances)), importances[indices], align='center')
# plt.xticks(range(len(importances)), np.array(feature_names)[indices], rotation=90)
# plt.xlim([-1, len(importances)])
# plt.tight_layout()
# plt.show()


# cmap = sns.color_palette("husl", 2)  # クラス数に応じたカラーパレット
# sns.set(style="whitegrid")

# # # 特徴量の散布図を描画
# plt.figure(figsize=(10, 6))

# # methodが'poly'の場合は以下のコードを使用
# # sns.scatterplot(x=X['acc_x_coef2'], y=X['acc_y_coef0'], hue=y, palette=cmap, alpha=0.5)
# # plt.legend(title='Class', loc='upper right')
# # plt.title('Feature Scatter Plot')
# # plt.xlabel('acc_x_coef2')
# # plt.ylabel('acc_y_coef0')
# # plt.grid()
# # plt.show()



# # # methodが'ar'の場合は以下のコードを使用
# # sns.scatterplot(x=X['acc_y_ar0'], y=X['acc_x_ar2'], hue=y, palette=cmap, alpha=0.5)
# # plt.legend(title='Class', loc='upper right')
# # plt.title('Feature Scatter Plot')
# # plt.xlabel('acc_y_ar0')
# # plt.ylabel('acc_x_ar2')
# # plt.grid()
# # plt.show()




In [ ]:
X.head()

In [ ]:
def predict_on_segments(model, df_predict, method='poly', degree_or_lags=3, 
                        segment_length=50, step=50):
    """
    予測対象のデータに対してセグメントごとにベクトル化し、学習済みモデルでクラス0かを判定
    
    Parameters:
    - model: 学習済みランダムフォレストモデル
    - df_predict: 予測対象のDataFrame（class列は不要）
    - method: 'poly' or 'ar'
    - degree_or_lags: 多項式次数 or ARラグ
    - segment_length: セグメントの長さ
    - step: セグメントのステップ幅

    Returns:
    - pred_labels: 各セグメントの予測ラベル（0 or 1）
    - pred_probs: クラス1（=元のクラス）の確率
    - segment_starts: 各セグメントの開始インデックス
    """
    feature_list = []
    segment_starts = []

    for start in range(0, len(df_predict) - segment_length + 1, step):
        segment = df_predict.iloc[start:start + segment_length]
        vec = vectorize_sequence(segment, method=method, degree_or_lags=degree_or_lags)
        feature_list.append(vec)
        segment_starts.append(start)

    df_vec = pd.DataFrame(feature_list)
    pred_probs = model.predict_proba(df_vec)  # クラスの確率
    pred_labels = model.predict(df_vec)

    return pred_labels, pred_probs, segment_starts,df_vec


## ランダムに誤差を追加して、精度確認

In [ ]:
test_start = 0
test_last = 100

df_test = df.iloc[test_start:test_last].copy()
df_test = df_test.drop(columns='class')
print(len(df_test))

In [ ]:
# df_test = pd.read_csv('C:/Users/hiros/OneDrive - 学校法人立命館/デスクトップ/TWILITE data/data1/acc_test.csv')

In [ ]:
# 予測データ: df_test（sequence_counter, acc_x などを含む）
pred_labels, pred_probs, segment_starts,df_vec = predict_on_segments(
    model=model,
    df_predict=df_test,
    method=method,
    degree_or_lags=degree_or_lags,
    segment_length=segment_length,  # 10以下ならdf_testに対応
    step=step
)

print("予測ラベルのサンプル:", pred_labels)
print("予測確率のサンプル:", pred_probs)
print("セグメント開始インデックスのサンプル:", segment_starts)
df_vec

In [ ]:
from sklearn.metrics import accuracy_score

# 正確な代表ラベル作成方法（例：多数決）
# テストデータ（df_test）に対応する正解ラベルを作成
test_labels = df.iloc[0:100]['class'].fillna(-1).astype(int).reset_index(drop=True)
segment_labels = []
for start in range(0, len(test_labels) -segment_length+1, step):
    segment = test_labels.iloc[start:start + segment_length]
    # NaN（ここでは-1にしたもの）は除外してmodeを取る
    segment_valid = segment[segment != -1]
    if not segment_valid.empty:
        label = segment_valid.mode().iloc[0]  # 最頻値
    else:
        label = -1  # すべてNaNの場合
    segment_labels.append(label)

# segment_labels = [1 if lbl == 0 else 0 for lbl in segment_labels]  #マルチクラス分類の場合、コメントアウト


print("segment_labels:", segment_labels)
print("pred_labels:", pred_labels)
print("len(segment_labels):", len(segment_labels))
print("len(pred_labels):", len(pred_labels))
accuracy_score(segment_labels, pred_labels)


In [ ]:
segment_labels = np.array(segment_labels)
pred_labels = np.array(pred_labels)

# 混同行列の表示
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

classes =  np.unique(df['class'].dropna()).astype(int) # クラスを取得
cm = confusion_matrix(segment_labels, pred_labels)

class_names = [f'Class {i}' for i in classes]

# plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()


print("教師ラベル:", segment_labels)
print("予測ラベル:", pred_labels)

ARモデルでの近似より、多項式での近似のほうが精度がいい

In [ ]:

from sklearn.tree import plot_tree
# 決定木の可視化
plt.figure(figsize=(20, 10))
plot_tree(model, feature_names=feature_names, filled=True, rounded=True)
plt.title('Decision Tree for Class 0 vs Others')
plt.show()


In [ ]:
# plt.figure(figsize=(10, 6))
# # 0番目の決定木を可視化（他の木を見たい場合はインデックスを変更）
# plot_tree(model.estimators_[0], feature_names=feature_names, filled=True, fontsize=10)
# plt.title('Decision Tree for Class 0 vs Others (Tree 0)')
# plt.show()